<a href="https://colab.research.google.com/github/FatimaZahraBoukab/HPC-Assignment3-CUDA/blob/main/Assignment_3_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
!nvcc --version

Sat Nov 21 22:03:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile ex1.cu
#include <stdio.h>

__global__ void hello() {
    printf("my cuda course | blockIdx.x = %d, threadIdx.x = %d\n", blockIdx.x, threadIdx.x);
}

int main() {
    hello<<<3, 5>>>();          // 3 blocs de 5 threads = 15 threads
    cudaDeviceSynchronize();
    return 0;
}

Writing ex1.cu


In [ ]:
!nvcc ex1.cu -o ex1
!./ex1

my cuda course | blockIdx.x = 2, threadIdx.x = 0
my cuda course | blockIdx.x = 2, threadIdx.x = 1
my cuda course | blockIdx.x = 2, threadIdx.x = 2
my cuda course | blockIdx.x = 2, threadIdx.x = 3
my cuda course | blockIdx.x = 2, threadIdx.x = 4
my cuda course | blockIdx.x = 0, threadIdx.x = 0
my cuda course | blockIdx.x = 0, threadIdx.x = 1
my cuda course | blockIdx.x = 0, threadIdx.x = 2
my cuda course | blockIdx.x = 0, threadIdx.x = 3
my cuda course | blockIdx.x = 0, threadIdx.x = 4
my cuda course | blockIdx.x = 1, threadIdx.x = 0
my cuda course | blockIdx.x = 1, threadIdx.x = 1
my cuda course | blockIdx.x = 1, threadIdx.x = 2
my cuda course | blockIdx.x = 1, threadIdx.x = 3
my cuda course | blockIdx.x = 1, threadIdx.x = 4


In [ ]:
%%writefile ex2.cu
#include <stdio.h>
#include <stdlib.h>

__global__ void fill(int *a, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // indice global du thread
    if (i < n) a[i] = i * i;
}

int main() {
    int threads = 256;

    // ---- Partie 1 : vérification des calculs ----
    int Ns[3] = {256, 257, 1000};
    for (int k = 0; k < 3; k++) {
        int N = Ns[k];
        int blocks = (N + threads - 1) / threads;
        int launched = blocks * threads;
        printf("N=%4d : blocs=%d, threads lances=%d, threads inactifs=%d\n",
               N, blocks, launched, launched - N);
    }

    // ---- Partie 2 : N = 1000 ----
    int N = 1000;
    int blocks = (N + threads - 1) / threads;
    int *h = (int *)malloc(N * sizeof(int));   // tableau sur l'hôte (CPU)
    int *d;                                    // tableau sur le device (GPU)
    cudaMalloc(&d, N * sizeof(int));

    fill<<<blocks, threads>>>(d, N);
    cudaMemcpy(h, d, N * sizeof(int), cudaMemcpyDeviceToHost);   // GPU -> CPU

    for (int i = N - 5; i < N; i++) printf("a[%d] = %d\n", i, h[i]);

    cudaFree(d);
    free(h);
    return 0;
}

Writing ex2.cu


In [ ]:
!nvcc ex2.cu -o ex2
!./ex2

N= 256 : blocs=1, threads lances=256, threads inactifs=0
N= 257 : blocs=2, threads lances=512, threads inactifs=255
N=1000 : blocs=4, threads lances=1024, threads inactifs=24
a[995] = 990025
a[996] = 992016
a[997] = 994009
a[998] = 996004
a[999] = 998001


In [ ]:
%%writefile ex3.cu
#include <stdio.h>
#include <stdlib.h>

__global__ void vecAdd(const int *a, const int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    const int N = 1000, threads = 256;
    int blocks = (N + threads - 1) / threads;
    size_t bytes = N * sizeof(int);

    // 1) Tableaux sur l'hôte, initialisés : a[i] = i, b[i] = 2*i
    int *ha = (int *)malloc(bytes);
    int *hb = (int *)malloc(bytes);
    int *hc = (int *)malloc(bytes);
    for (int i = 0; i < N; i++) { ha[i] = i; hb[i] = 2 * i; }

    // 2) Les 3 tableaux sur le GPU
    int *da, *db, *dc;
    cudaMalloc(&da, bytes);
    cudaMalloc(&db, bytes);
    cudaMalloc(&dc, bytes);

    // 3) Les 2 copies hôte -> device
    cudaMemcpy(da, ha, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(db, hb, bytes, cudaMemcpyHostToDevice);

    // 4) Lancement du kernel
    vecAdd<<<blocks, threads>>>(da, db, dc, N);

    // 5) La copie device -> hôte
    cudaMemcpy(hc, dc, bytes, cudaMemcpyDeviceToHost);

    // 6) Vérification sur l'hôte : c[i] == 3*i pour tout i
    int bad = -1;
    for (int i = 0; i < N; i++) {
        if (hc[i] != 3 * i) { bad = i; break; }
    }
    if (bad < 0) printf("OK\n");
    else printf("Erreur a l'indice %d : c=%d, attendu=%d\n", bad, hc[bad], 3 * bad);

    // 7) Libérer toute la mémoire
    cudaFree(da); cudaFree(db); cudaFree(dc);
    free(ha); free(hb); free(hc);
    return 0;
}

Writing ex3.cu


In [ ]:
!nvcc ex3.cu -o ex3
!./ex3

OK


In [ ]:
%%writefile ex4.cu
#include <stdio.h>
#include <stdlib.h>

__global__ void mulInPlace(const float *x, float *y, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = x[i] * y[i];     // le résultat écrase y
}

int main() {
    const int N = 100, threads = 30;
    int blocks = (N + threads - 1) / threads;
    printf("blocs=%d, threads lances=%d, inactifs=%d\n",
           blocks, blocks * threads, blocks * threads - N);

    size_t bytes = N * sizeof(float);
    float *hx = (float *)malloc(bytes);
    float *hy = (float *)malloc(bytes);
    for (int i = 0; i < N; i++) { hx[i] = (float)i; hy[i] = 2.0f; }

    float *dx, *dy;                    // 2 tableaux GPU
    cudaMalloc(&dx, bytes);
    cudaMalloc(&dy, bytes);
    cudaMemcpy(dx, hx, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dy, hy, bytes, cudaMemcpyHostToDevice);

    mulInPlace<<<blocks, threads>>>(dx, dy, N);

    cudaMemcpy(hy, dy, bytes, cudaMemcpyDeviceToHost);
    for (int i = 0; i < N; i++) printf("Y[%d] = %.1f\n", i, hy[i]);

    cudaFree(dx); cudaFree(dy);
    free(hx); free(hy);
    return 0;
}

Writing ex4.cu


In [ ]:
!nvcc ex4.cu -o ex4
!./ex4

blocs=4, threads lances=120, inactifs=20
Y[0] = 0.0
Y[1] = 2.0
Y[2] = 4.0
Y[3] = 6.0
Y[4] = 8.0
Y[5] = 10.0
Y[6] = 12.0
Y[7] = 14.0
Y[8] = 16.0
Y[9] = 18.0
Y[10] = 20.0
Y[11] = 22.0
Y[12] = 24.0
Y[13] = 26.0
Y[14] = 28.0
Y[15] = 30.0
Y[16] = 32.0
Y[17] = 34.0
Y[18] = 36.0
Y[19] = 38.0
Y[20] = 40.0
Y[21] = 42.0
Y[22] = 44.0
Y[23] = 46.0
Y[24] = 48.0
Y[25] = 50.0
Y[26] = 52.0
Y[27] = 54.0
Y[28] = 56.0
Y[29] = 58.0
Y[30] = 60.0
Y[31] = 62.0
Y[32] = 64.0
Y[33] = 66.0
Y[34] = 68.0
Y[35] = 70.0
Y[36] = 72.0
Y[37] = 74.0
Y[38] = 76.0
Y[39] = 78.0
Y[40] = 80.0
Y[41] = 82.0
Y[42] = 84.0
Y[43] = 86.0
Y[44] = 88.0
Y[45] = 90.0
Y[46] = 92.0
Y[47] = 94.0
Y[48] = 96.0
Y[49] = 98.0
Y[50] = 100.0
Y[51] = 102.0
Y[52] = 104.0
Y[53] = 106.0
Y[54] = 108.0
Y[55] = 110.0
Y[56] = 112.0
Y[57] = 114.0
Y[58] = 116.0
Y[59] = 118.0
Y[60] = 120.0
Y[61] = 122.0
Y[62] = 124.0
Y[63] = 126.0
Y[64] = 128.0
Y[65] = 130.0
Y[66] = 132.0
Y[67] = 134.0
Y[68] = 136.0
Y[69] = 138.0
Y[70] = 140.0
Y[71] = 142.0
Y[72] = 144.0
Y[

In [ ]:
%%writefile ex5.cu
#include <stdio.h>
#include <stdlib.h>

#define ROWS 20
#define COLS 40

__global__ void fillMatrix(int *A, int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;   // x -> colonnes
    int row = blockIdx.y * blockDim.y + threadIdx.y;   // y -> lignes
    if (row < rows && col < cols)
        A[row * cols + col] = 100 * row + col;         // stockage row-major
}

int main() {
    dim3 block(16, 16);
    dim3 grid((COLS + block.x - 1) / block.x,          // 3
              (ROWS + block.y - 1) / block.y);         // 2
    int launched = grid.x * grid.y * block.x * block.y;
    printf("grille = (%d,%d), threads lances=%d, inactifs=%d\n",
           grid.x, grid.y, launched, launched - ROWS * COLS);

    size_t bytes = ROWS * COLS * sizeof(int);
    int *h = (int *)malloc(bytes);
    int *d;
    cudaMalloc(&d, bytes);

    fillMatrix<<<grid, block>>>(d, ROWS, COLS);
    cudaMemcpy(h, d, bytes, cudaMemcpyDeviceToHost);

    for (int r = 0; r < ROWS; r++) {
        for (int c = 0; c < COLS; c++) printf("%5d", h[r * COLS + c]);
        printf("\n");
    }
    printf("A[17][33] = %d\n", h[17 * COLS + 33]);

    cudaFree(d);
    free(h);
    return 0;
}

Writing ex5.cu


In [ ]:
!nvcc ex5.cu -o ex5
!./ex5

grille = (3,2), threads lances=1536, inactifs=736
    0    1    2    3    4    5    6    7    8    9   10   11   12   13   14   15   16   17   18   19   20   21   22   23   24   25   26   27   28   29   30   31   32   33   34   35   36   37   38   39
  100  101  102  103  104  105  106  107  108  109  110  111  112  113  114  115  116  117  118  119  120  121  122  123  124  125  126  127  128  129  130  131  132  133  134  135  136  137  138  139
  200  201  202  203  204  205  206  207  208  209  210  211  212  213  214  215  216  217  218  219  220  221  222  223  224  225  226  227  228  229  230  231  232  233  234  235  236  237  238  239
  300  301  302  303  304  305  306  307  308  309  310  311  312  313  314  315  316  317  318  319  320  321  322  323  324  325  326  327  328  329  330  331  332  333  334  335  336  337  338  339
  400  401  402  403  404  405  406  407  408  409  410  411  412  413  414  415  416  417  418  419  420  421  422  423  424  425  426  427  428 